# 06. Regression Discontinuity

## Background

Regression Discontinuity is one of the most credible quasi-experimental designs because the identification assumption is local and transparent: units just below a threshold are a valid counterfactual for units just above it. The "as-if random" variation comes from the discontinuity in treatment assignment, not from any global ignorability assumption.

The canonical example: students with test scores just above a cutoff get a scholarship; those just below don't. Near the cutoff, whether a student lands above or below is essentially random (measurement noise, rounding). This local randomization licenses a causal interpretation of the jump in outcomes at the threshold.

## What You'll Learn

- Sharp RD: treatment is a deterministic function of the running variable
- Fuzzy RD: treatment probability jumps at the cutoff (but not 0→1)
- Bandwidth selection: the bias-variance tradeoff around the cutoff
- Local polynomial estimation: why linear extrapolation to the cutoff is standard
- RD validity checks: manipulation test (McCrary density test), covariate smoothness

## Why This Matters

RD has become a workhorse of applied economics and program evaluation (Lee & Lemieux 2010). It requires only that the running variable is continuous and no one perfectly manipulates their score to land just above/below the cutoff. The `rdrobust` package (Calonico, Cattaneo & Titiunik 2014) implements optimal bandwidth selection and bias-corrected confidence intervals — the community standard for RD inference.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Sharp RD — Simulation

In [ ]:
def simulate_sharp_rd(n=2000, cutoff=0.0, true_effect=5.0, seed=0):
    """
    Sharp RD: T = 1[X >= cutoff].
    Outcome has a smooth (quadratic) relationship with X plus a jump at cutoff.
    """
    rng = np.random.default_rng(seed)
    # Running variable: centered at cutoff
    X = rng.uniform(-1, 1, n)
    # Treatment: sharp at cutoff
    T = (X >= cutoff).astype(int)
    # Potential outcomes: quadratic in X, different slopes each side
    y0 = 2 + 3*X - 1.5*X**2 + rng.normal(0, 1.5, n)
    y1 = y0 + true_effect
    Y  = np.where(T, y1, y0)
    return pd.DataFrame({'X': X, 'T': T, 'Y': Y, 'y0': y0, 'y1': y1})


df_rd = simulate_sharp_rd(n=2000, true_effect=5.0)
print(f"n={len(df_rd)}, treated above cutoff n={df_rd.T.sum()}")
print(f"True effect at cutoff = 5.0")

In [ ]:
def plot_rd(df, cutoff=0.0, bandwidth=None, n_bins=30, title='RD Plot'):
    """Binned scatter plot with local regression lines each side."""
    fig, ax = plt.subplots(figsize=(9, 5))

    if bandwidth:
        df = df[(df.X >= cutoff - bandwidth) & (df.X <= cutoff + bandwidth)]

    # Bin the running variable
    df = df.copy()
    df['bin'] = pd.cut(df.X, bins=n_bins)
    binned = df.groupby('bin', observed=True).agg(
        x_mid=('X', 'mean'), y_mean=('Y', 'mean'), n=('Y', 'count')
    ).dropna()

    left  = binned[binned.x_mid < cutoff]
    right = binned[binned.x_mid >= cutoff]

    ax.scatter(left.x_mid,  left.y_mean,  s=left.n*0.5,  color='#2196F3', alpha=0.8, label='Control')
    ax.scatter(right.x_mid, right.y_mean, s=right.n*0.5, color='#F44336', alpha=0.8, label='Treated')

    # Fit local lines (linear on each side)
    for side_df, color in [(df[df.X < cutoff], '#2196F3'), (df[df.X >= cutoff], '#F44336')]:
        if len(side_df) > 5:
            m = smf.ols('Y ~ X + I(X**2)', data=side_df).fit()
            x_grid = np.linspace(side_df.X.min(), side_df.X.max(), 100)
            y_hat  = m.predict(pd.DataFrame({'X': x_grid}))
            ax.plot(x_grid, y_hat, color=color, lw=2.5)

    ax.axvline(cutoff, color='black', ls='--', lw=2, label=f'Cutoff = {cutoff}')
    ax.set_xlabel('Running variable (X)'); ax.set_ylabel('Outcome (Y)')
    ax.set_title(title); ax.legend()
    return fig


fig = plot_rd(df_rd, title='Sharp RD — Raw Binned Scatter')
plt.tight_layout()
plt.savefig('06_rd_scatter.png', dpi=110, bbox_inches='tight')
plt.show()

## 2. Local Linear Estimation

Estimate the treatment effect as the difference in intercepts from local linear regressions on each side of the cutoff. Use only observations within a bandwidth h of the cutoff.

In [ ]:
def local_linear_rd(df, cutoff=0.0, bandwidth=0.5, kernel='triangular'):
    """
    Local linear RD estimator.
    kernel: 'triangular' (standard) or 'uniform'
    """
    df_h = df[(df.X >= cutoff - bandwidth) & (df.X <= cutoff + bandwidth)].copy()
    df_h['xc'] = df_h.X - cutoff   # center at cutoff

    if kernel == 'triangular':
        df_h['w'] = 1 - np.abs(df_h.xc) / bandwidth
    else:
        df_h['w'] = 1.0

    # Local linear: Y ~ xc + T + T*xc (allows different slopes)
    m = smf.wls('Y ~ xc + T + T:xc', data=df_h, weights=df_h.w).fit()
    tau  = m.params['T']
    se   = m.bse['T']
    ci   = (tau - 1.96*se, tau + 1.96*se)
    return dict(tau=tau, se=se, ci=ci, n=len(df_h), bandwidth=bandwidth)


# Effect across different bandwidths
bws = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
results = [local_linear_rd(df_rd, bandwidth=h) for h in bws]

fig, ax = plt.subplots(figsize=(8, 4))
taus = [r['tau'] for r in results]
cis  = [r['ci']  for r in results]
ax.plot(bws, taus, 'o-', color='#2196F3', lw=2, label='RD estimate')
ax.fill_between(bws, [c[0] for c in cis], [c[1] for c in cis], alpha=0.2, color='#2196F3')
ax.axhline(5.0, color='red', ls='--', lw=2, label='True effect = 5.0')
ax.set_xlabel('Bandwidth h'); ax.set_ylabel('Estimated effect')
ax.set_title('Bandwidth Sensitivity — Bias-Variance Tradeoff')
ax.legend()
plt.tight_layout()
plt.savefig('06_bandwidth.png', dpi=110, bbox_inches='tight')
plt.show()

print(f"{'Bandwidth':>10} {'Estimate':>10} {'SE':>8} {'n in window':>12}")
for h, r in zip(bws, results):
    print(f"{h:>10.2f} {r['tau']:>10.3f} {r['se']:>8.3f} {r['n']:>12}")

## 3. Fuzzy RD

When treatment doesn't jump from 0 to 1 at the cutoff but the *probability* of treatment jumps, we have a fuzzy RD. The estimator is a ratio: (jump in outcome) / (jump in treatment probability) — essentially a local IV estimator.

In [ ]:
def simulate_fuzzy_rd(n=2000, cutoff=0.0, true_late=6.0, seed=0):
    """
    Fuzzy RD: P(T=1|X) jumps from ~0.2 to ~0.8 at cutoff.
    Captures the LATE for compliers near the threshold.
    """
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, n)
    # Assignment rule: above cutoff more likely treated, but not deterministic
    p_treat = np.where(X >= cutoff, 0.80, 0.20)
    T = rng.binomial(1, p_treat)
    y0 = 2 + 3*X + rng.normal(0, 2, n)
    y1 = y0 + true_late
    Y  = np.where(T, y1, y0)
    return pd.DataFrame({'X': X, 'T': T, 'Y': Y})


df_fuzzy = simulate_fuzzy_rd(n=2000, true_late=6.0)

def fuzzy_rd_estimate(df, cutoff=0.0, bandwidth=0.4):
    """Fuzzy RD via local Wald ratio (reduced form / first stage)."""
    df_h = df[(df.X >= cutoff - bandwidth) & (df.X <= cutoff + bandwidth)].copy()
    df_h['xc'] = df_h.X - cutoff
    df_h['w'] = 1 - np.abs(df_h.xc) / bandwidth

    # Reduced form: effect of crossing cutoff on Y
    m_rf = smf.wls('Y ~ xc + I(X >= @cutoff) + I(X >= @cutoff):xc',
                   data=df_h, weights=df_h.w).fit()
    # First stage: effect of crossing cutoff on T
    m_fs = smf.wls('T ~ xc + I(X >= @cutoff) + I(X >= @cutoff):xc',
                   data=df_h, weights=df_h.w).fit()

    jump_y = m_rf.params['I(X >= @cutoff)[True]']
    jump_t = m_fs.params['I(X >= @cutoff)[True]']
    late   = jump_y / jump_t
    return dict(late=late, jump_y=jump_y, jump_t=jump_t)


res_fuzzy = fuzzy_rd_estimate(df_fuzzy, bandwidth=0.4)
print(f"Fuzzy RD Results (bandwidth=0.4):")
print(f"  Reduced form (jump in Y): {res_fuzzy['jump_y']:.3f}")
print(f"  First stage  (jump in T): {res_fuzzy['jump_t']:.3f}")
print(f"  LATE estimate:            {res_fuzzy['late']:.3f}")
print(f"  True LATE:                6.000")

## 4. Validity Checks

**Manipulation test (McCrary density test)**: If units can manipulate their score to land just above the cutoff, the density of the running variable will have a discontinuity at the cutoff. A smooth density is a necessary condition for RD validity.

**Covariate smoothness**: Pre-determined covariates should be continuous at the cutoff. A jump in baseline covariates suggests units near the cutoff differ systematically — possibly due to manipulation or misspecification.

In [ ]:
def mccrary_density_test(X, cutoff=0.0, bandwidth=0.5, n_bins=40):
    """
    Visual and statistical McCrary density test.
    Fit separate linear models to the histogram density on each side.
    A significant jump suggests manipulation.
    """
    # Bin counts near cutoff
    X_h = X[(X >= cutoff - bandwidth) & (X <= cutoff + bandwidth)]
    counts, edges = np.histogram(X_h, bins=n_bins)
    mids = (edges[:-1] + edges[1:]) / 2
    width = edges[1] - edges[0]
    density = counts / (len(X) * width)

    df_d = pd.DataFrame({'x': mids, 'density': density})
    df_d['above'] = (df_d.x >= cutoff).astype(int)
    df_d['xc'] = df_d.x - cutoff

    # Fit local linear on each side
    m_l = smf.ols('density ~ xc', data=df_d[df_d.above==0]).fit()
    m_r = smf.ols('density ~ xc', data=df_d[df_d.above==1]).fit()

    pred_l = m_l.predict(pd.DataFrame({'xc': [0 - 1e-6]})).iloc[0]
    pred_r = m_r.predict(pd.DataFrame({'xc': [0 + 1e-6]})).iloc[0]
    jump   = pred_r - pred_l

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(mids, density, width=width*0.9, color=np.where(mids<cutoff,'#2196F3','#F44336'), alpha=0.6)
    x_l = np.linspace(cutoff - bandwidth, cutoff, 50)
    x_r = np.linspace(cutoff, cutoff + bandwidth, 50)
    ax.plot(x_l, m_l.predict(pd.DataFrame({'xc': x_l - cutoff})), 'b-', lw=2.5)
    ax.plot(x_r, m_r.predict(pd.DataFrame({'xc': x_r - cutoff})), 'r-', lw=2.5)
    ax.axvline(cutoff, color='black', ls='--', lw=2)
    ax.set_xlabel('Running variable'); ax.set_ylabel('Density')
    ax.set_title(f'McCrary Density Test — Jump at cutoff = {jump:.4f}')
    plt.tight_layout()
    plt.savefig('06_mccrary.png', dpi=110, bbox_inches='tight')
    plt.show()
    print(f"Density jump at cutoff: {jump:.4f}  (near 0 = no manipulation)")
    return jump


jump = mccrary_density_test(df_rd.X.values, cutoff=0.0)

# Covariate smoothness test (run RD on pre-treatment covariates)
print("\nCovariate smoothness check (should be near 0 if valid):")
# Simulate pre-treatment covariate
np.random.seed(42)
df_rd['age'] = 45 + 5*df_rd.X + np.random.normal(0, 3, len(df_rd))  # smooth
res_age = local_linear_rd(df_rd.rename(columns={'age':'Y_check'}), bandwidth=0.4)
print(f"  Age (pre-treatment covariate): jump = {res_age['tau']:.3f}  "
      f"(SE={res_age['se']:.3f}) {'✓ smooth' if abs(res_age['tau']) < 1 else '✗ suspicious jump!'}")